# NB_bishop_ch06_deep_networks

**Bishop Chapter 6 — Deep Networks: Universal Approximation, Activations & Mixture Density Networks**

This notebook demonstrates key concepts from Bishop Chapter 6: building MLPs in Keras, the universal approximation theorem, comparing activation functions, and mixture density networks for multi-valued mappings.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_06/NB_bishop_ch06_deep_networks.ipynb)

In [ ]:
# Install dependencies (Colab-safe)
# !pip install tensorflow numpy matplotlib

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version: {np.__version__}')

In [ ]:
# Chart styling setup
matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

## Part 1: Universal Approximation Theorem

The universal approximation theorem states that a feedforward network with a single hidden layer containing a finite number of neurons can approximate any continuous function on compact subsets of $\mathbb{R}^n$. We demonstrate this by fitting an arbitrary complex function.

In [ ]:
# Define a complex target function
np.random.seed(42)
tf.random.set_seed(42)

def target_function(x):
    """A complex function combining oscillations and nonlinearity."""
    return np.sin(2 * x) * np.exp(-0.1 * x**2) + 0.3 * np.cos(5 * x)

x_train = np.linspace(-4, 4, 500).astype(np.float32).reshape(-1, 1)
y_train = target_function(x_train).astype(np.float32)

print(f'Training data: x shape {x_train.shape}, y shape {y_train.shape}')
print(f'y range: [{y_train.min():.3f}, {y_train.max():.3f}]')

In [ ]:
# Build networks with increasing hidden units to show approximation capability
hidden_sizes = [2, 5, 20, 100]
models = {}
histories = {}

for h in hidden_sizes:
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(h, activation='relu', input_shape=(1,)),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    hist = model.fit(x_train, y_train, epochs=500, verbose=0, batch_size=32)
    models[h] = model
    histories[h] = hist
    print(f'Hidden units = {h:>3d}: final loss = {hist.history["loss"][-1]:.6f}')

In [ ]:
# Plot universal approximation results
colors = ['#cd0000', '#DAA520', '#228B22', '#1a3a6e']

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for idx, h in enumerate(hidden_sizes):
    y_pred = models[h].predict(x_train, verbose=0)
    axes[idx].plot(x_train, y_train, 'k-', linewidth=1.5, label='Target', alpha=0.6)
    axes[idx].plot(x_train, y_pred, color=colors[idx], linewidth=2, label=f'{h} units')
    axes[idx].set_title(f'{h} hidden units')
    axes[idx].set_xlabel('x')
    axes[idx].legend()
axes[0].set_ylabel('f(x)')
fig.suptitle('Universal Approximation: Increasing Network Width', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'bishop_ch6_universal_approx')
plt.show()

## Part 2: Activation Function Comparison

Different activation functions introduce different nonlinearities. We compare sigmoid, tanh, and ReLU on the same regression task.

In [ ]:
# Visualize the activation functions and their derivatives
x_act = np.linspace(-5, 5, 500).astype(np.float32)

act_fns = {
    'Sigmoid': (tf.nn.sigmoid, lambda x: tf.nn.sigmoid(x) * (1 - tf.nn.sigmoid(x))),
    'Tanh': (tf.nn.tanh, lambda x: 1 - tf.nn.tanh(x)**2),
    'ReLU': (tf.nn.relu, lambda x: tf.cast(x > 0, tf.float32))
}

colors_act = {'Sigmoid': '#1a3a6e', 'Tanh': '#cd0000', 'ReLU': '#228B22'}

In [ ]:
# Train identical architectures with different activations
act_models = {}
act_histories = {}

for act_name in ['sigmoid', 'tanh', 'relu']:
    tf.random.set_seed(42)
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(50, activation=act_name, input_shape=(1,)),
        tf.keras.layers.Dense(50, activation=act_name),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='mse')
    hist = model.fit(x_train, y_train, epochs=300, verbose=0, batch_size=32)
    act_models[act_name] = model
    act_histories[act_name] = hist
    print(f'{act_name:>8s}: final loss = {hist.history["loss"][-1]:.6f}')

In [ ]:
# Plot activation comparison: function shapes and learning curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Panel 1: Activation function shapes
for name, (fn, _) in act_fns.items():
    axes[0].plot(x_act, fn(x_act).numpy(), label=name, color=colors_act[name], linewidth=2)
axes[0].axhline(0, color='gray', lw=0.5, ls='--')
axes[0].axvline(0, color='gray', lw=0.5, ls='--')
axes[0].set_title('Activation Functions')
axes[0].set_xlabel('z')
axes[0].set_ylabel('g(z)')
axes[0].legend()

# Panel 2: Training loss comparison
for act_name, c in zip(['sigmoid', 'tanh', 'relu'], ['#1a3a6e', '#cd0000', '#228B22']):
    axes[1].plot(act_histories[act_name].history['loss'], label=act_name, color=c, linewidth=1.5)
axes[1].set_title('Training Loss by Activation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE')
axes[1].set_yscale('log')
axes[1].legend()

# Panel 3: Predictions overlay
axes[2].plot(x_train, y_train, 'k-', lw=1.5, label='Target', alpha=0.5)
for act_name, c in zip(['sigmoid', 'tanh', 'relu'], ['#1a3a6e', '#cd0000', '#228B22']):
    y_pred = act_models[act_name].predict(x_train, verbose=0)
    axes[2].plot(x_train, y_pred, label=act_name, color=c, linewidth=1.5)
axes[2].set_title('Predictions by Activation')
axes[2].set_xlabel('x')
axes[2].set_ylabel('f(x)')
axes[2].legend()

fig.tight_layout()
save_fig(fig, 'bishop_ch6_activations')
plt.show()

### Vanishing Gradient Problem

Sigmoid and tanh saturate for large inputs, causing gradients to vanish. Let us measure gradient magnitudes across layers.

In [ ]:
# Measure gradient norms per layer for each activation
for act_name in ['sigmoid', 'tanh', 'relu']:
    tf.random.set_seed(42)
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(50, activation=act_name, input_shape=(1,)),
        tf.keras.layers.Dense(50, activation=act_name),
        tf.keras.layers.Dense(50, activation=act_name),
        tf.keras.layers.Dense(50, activation=act_name),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    x_batch = x_train[:32]
    y_batch = y_train[:32]
    with tf.GradientTape() as tape:
        pred = model(x_batch, training=True)
        loss = tf.reduce_mean((pred - y_batch)**2)
    grads = tape.gradient(loss, model.trainable_variables)
    norms = [tf.norm(g).numpy() for g in grads if len(g.shape) == 2]
    print(f'{act_name:>8s} layer gradient norms: {[f"{n:.4f}" for n in norms]}')

## Part 3: Mixture Density Network (MDN)

Standard networks map each input to a single output. For multi-valued functions (one-to-many mappings), we need a network that outputs a mixture of Gaussians. This is Bishop's mixture density network.

In [ ]:
# Generate inverse-sine data: multi-valued mapping
np.random.seed(42)
N = 1000
t = np.random.uniform(0, 1, N).astype(np.float32)
x_mdn = t + 0.3 * np.sin(2 * np.pi * t) + np.random.normal(0, 0.03, N).astype(np.float32)

# Swap x and t to create multi-valued mapping: x -> t
x_input = x_mdn.reshape(-1, 1)
y_target = t.reshape(-1, 1)

print(f'MDN input shape: {x_input.shape}')
print(f'MDN target shape: {y_target.shape}')

In [ ]:
# Define MDN model
K = 5  # number of mixture components

inputs = tf.keras.Input(shape=(1,))
h = tf.keras.layers.Dense(64, activation='relu')(inputs)
h = tf.keras.layers.Dense(64, activation='relu')(h)

# Output: pi (mixing coefficients), mu (means), sigma (std devs)
pi_out = tf.keras.layers.Dense(K, activation='softmax', name='pi')(h)
mu_out = tf.keras.layers.Dense(K, name='mu')(h)
sigma_out = tf.keras.layers.Dense(K, activation='softplus', name='sigma')(h)

mdn_model = tf.keras.Model(inputs=inputs, outputs=[pi_out, mu_out, sigma_out])
mdn_model.summary()

In [ ]:
# MDN negative log-likelihood loss
def mdn_loss(y_true, pi, mu, sigma):
    """Negative log-likelihood of mixture of Gaussians."""
    y_true = tf.expand_dims(y_true, -1)  # (batch, 1, 1)
    y_true = tf.squeeze(y_true, axis=1)   # (batch, 1)
    
    normal = tf.exp(-0.5 * ((y_true - mu) / sigma)**2) / (sigma * tf.sqrt(2.0 * np.pi))
    weighted = pi * normal
    mixture = tf.reduce_sum(weighted, axis=-1)
    nll = -tf.reduce_mean(tf.math.log(mixture + 1e-8))
    return nll

In [ ]:
# Train the MDN
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
mdn_losses = []

dataset = tf.data.Dataset.from_tensor_slices((x_input, y_target)).shuffle(1000).batch(64)

for epoch in range(300):
    epoch_loss = []
    for xb, yb in dataset:
        with tf.GradientTape() as tape:
            pi, mu, sigma = mdn_model(xb, training=True)
            loss = mdn_loss(yb, pi, mu, sigma)
        grads = tape.gradient(loss, mdn_model.trainable_variables)
        optimizer.apply_gradients(zip(grads, mdn_model.trainable_variables))
        epoch_loss.append(loss.numpy())
    mdn_losses.append(np.mean(epoch_loss))
    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch+1:>3d}: loss = {mdn_losses[-1]:.4f}')

In [ ]:
# Sample from the MDN to visualize conditional density
x_test = np.linspace(x_input.min(), x_input.max(), 200).astype(np.float32).reshape(-1, 1)
pi_pred, mu_pred, sigma_pred = mdn_model.predict(x_test, verbose=0)

# Sample multiple outputs per input
samples_per_x = 10
x_plot, y_plot = [], []
for i in range(len(x_test)):
    for _ in range(samples_per_x):
        k = np.random.choice(K, p=pi_pred[i])
        y_sample = np.random.normal(mu_pred[i, k], sigma_pred[i, k])
        x_plot.append(x_test[i, 0])
        y_plot.append(y_sample)

In [ ]:
# Plot MDN results
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Panel 1: Original data
axes[0].scatter(x_input, y_target, s=3, alpha=0.3, color='#1a3a6e')
axes[0].set_title('Training Data (multi-valued)')
axes[0].set_xlabel('x')
axes[0].set_ylabel('t')

# Panel 2: MDN samples
axes[1].scatter(x_plot, y_plot, s=2, alpha=0.3, color='#cd0000')
axes[1].set_title('MDN Samples')
axes[1].set_xlabel('x')
axes[1].set_ylabel('t')

# Panel 3: Component means
for k_idx in range(K):
    axes[2].plot(x_test, mu_pred[:, k_idx], linewidth=1.5, label=f'$\\mu_{k_idx+1}$')
axes[2].scatter(x_input, y_target, s=1, alpha=0.1, color='gray')
axes[2].set_title('Mixture Component Means')
axes[2].set_xlabel('x')
axes[2].set_ylabel('t')
axes[2].legend()

fig.tight_layout()
save_fig(fig, 'bishop_ch6_mdn')
plt.show()

### MDN Conditional Density Visualization

In [ ]:
# Visualize conditional density at specific x values
x_slices = [0.3, 0.5, 0.7, 0.9]
t_range = np.linspace(-0.2, 1.2, 300)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for idx, x_val in enumerate(x_slices):
    x_q = np.array([[x_val]], dtype=np.float32)
    pi_q, mu_q, sigma_q = mdn_model.predict(x_q, verbose=0)
    density = np.zeros_like(t_range)
    for k_idx in range(K):
        comp = pi_q[0, k_idx] * np.exp(-0.5 * ((t_range - mu_q[0, k_idx]) / sigma_q[0, k_idx])**2) / (sigma_q[0, k_idx] * np.sqrt(2 * np.pi))
        density += comp
    axes[idx].fill_between(t_range, density, alpha=0.3, color='#1a3a6e')
    axes[idx].plot(t_range, density, color='#1a3a6e', linewidth=1.5)
    axes[idx].set_title(f'p(t | x={x_val})')
    axes[idx].set_xlabel('t')
axes[0].set_ylabel('Density')
fig.suptitle('MDN Conditional Densities', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## Depth vs Width Experiment

In [ ]:
# Compare: wide-shallow vs narrow-deep networks
configs = {
    'Wide-Shallow (1x200)': [200],
    'Medium (2x50)': [50, 50],
    'Deep-Narrow (5x20)': [20, 20, 20, 20, 20],
    'Very Deep (10x10)': [10]*10
}

config_results = {}
for name, layers in configs.items():
    tf.random.set_seed(42)
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(1,)))
    for units in layers:
        model.add(tf.keras.layers.Dense(units, activation='relu'))
    model.add(tf.keras.layers.Dense(1))
    model.compile(optimizer='adam', loss='mse')
    hist = model.fit(x_train, y_train, epochs=300, verbose=0, batch_size=32)
    config_results[name] = hist.history['loss']
    n_params = model.count_params()
    print(f'{name:>25s}: params={n_params:>6d}, final_loss={hist.history["loss"][-1]:.6f}')

In [ ]:
# Plot depth vs width comparison
fig, ax = plt.subplots(figsize=(8, 5))
colors_dw = ['#1a3a6e', '#cd0000', '#228B22', '#DAA520']
for (name, losses), c in zip(config_results.items(), colors_dw):
    ax.plot(losses, label=name, color=c, linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_yscale('log')
ax.set_title('Depth vs Width: Training Convergence')
ax.legend()
fig.tight_layout()
plt.show()

## Weight Initialization Effects

In [ ]:
# Compare weight initialization strategies
initializers = {
    'zeros': 'zeros',
    'random_normal': tf.keras.initializers.RandomNormal(stddev=0.5),
    'glorot_uniform': 'glorot_uniform',
    'he_normal': 'he_normal'
}

init_results = {}
for name, init in initializers.items():
    tf.random.set_seed(42)
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(50, activation='relu', kernel_initializer=init, input_shape=(1,)),
        tf.keras.layers.Dense(50, activation='relu', kernel_initializer=init),
        tf.keras.layers.Dense(1, kernel_initializer=init)
    ])
    model.compile(optimizer='adam', loss='mse')
    hist = model.fit(x_train, y_train, epochs=200, verbose=0, batch_size=32)
    init_results[name] = hist.history['loss']
    print(f'{name:>20s}: final_loss = {hist.history["loss"][-1]:.6f}')

In [ ]:
# Plot initialization comparison
fig, ax = plt.subplots(figsize=(8, 5))
for (name, losses), c in zip(init_results.items(), ['#cd0000', '#DAA520', '#1a3a6e', '#228B22']):
    ax.plot(losses, label=name, color=c, linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_yscale('log')
ax.set_title('Effect of Weight Initialization')
ax.legend()
fig.tight_layout()
plt.show()

## Summary

Key takeaways from Bishop Chapter 6:
- **Universal approximation**: A single hidden layer can approximate any continuous function, but more units / deeper networks learn faster in practice.
- **Activation functions**: ReLU avoids vanishing gradients and trains faster than sigmoid/tanh for deep networks.
- **Mixture density networks**: Enable modeling multi-valued (one-to-many) mappings by outputting parameters of a Gaussian mixture.
- **Depth vs width**: Deeper networks can be more parameter-efficient but may be harder to optimize.
- **Initialization**: Proper initialization (He, Glorot) is critical for training deep networks.

In [ ]:
print('Notebook complete.')